In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

In [2]:
# Define state
class LearningState(TypedDict):
    topic: str
    progress: str
    suggestion: str

In [3]:
# Node: track progress
def track_learning(state: LearningState):
    topic = state["topic"]

    return {
        "progress": f"You are currently learning: {topic}"
    }

In [4]:
# Node: suggest next step
def suggest_next(state: LearningState):
    topic = state["topic"]

    return {
        "suggestion": f"Next step: Practice problems on {topic}"
    }


In [5]:
# Build graph
graph = StateGraph(LearningState)

graph.add_node("track_learning", track_learning)
graph.add_node("suggest_next", suggest_next)

graph.add_edge(START, "track_learning")
graph.add_edge("track_learning", "suggest_next")
graph.add_edge("suggest_next", END)

In [6]:
# Add persistence
memory = MemorySaver()
workflow = graph.compile(checkpointer=memory)

# Thread ID (important)
config = {"configurable": {"thread_id": "student1"}}

In [7]:
# First run
result1 = workflow.invoke({"topic": "Machine Learning"}, config=config)
print(result1)


{'topic': 'Machine Learning', 'progress': 'You are currently learning: Machine Learning', 'suggestion': 'Next step: Practice problems on Machine Learning'}


In [8]:
# Second run (same thread → remembers flow)
result2 = workflow.invoke({"topic": "Deep Learning"}, config=config)
print(result2)

{'topic': 'Deep Learning', 'progress': 'You are currently learning: Deep Learning', 'suggestion': 'Next step: Practice problems on Deep Learning'}
